# Comparación de detectores, objetos y relaciones

YOLOv8n y YOLO26n se comparan sobre las mismas 2.619 imágenes de SUN RGB-D. El AP se calcula desde un umbral mínimo de confianza y la precisión, exhaustividad y F1 se calculan con el umbral operativo. Después se mide el efecto de la evidencia de objetos sobre las consultas compatibles. Visual Genome se utiliza únicamente para contrastar la cobertura de las reglas geométricas 2D.

In [1]:
from pathlib import Path
import sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'semantic_navigation_ws' / 'src').is_dir())
sys.path.insert(0, str(repo / 'experiments' / 'shared'))
import pandas as pd
from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from offline_benchmarks import (create_detector_figures, run_detector_benchmark,
                                write_results_summary)
ctx = bootstrap_offline()
print(f"Dispositivo: {ctx['device']}")
display(pd.DataFrame([{'detector': key, 'checkpoint': value}
                      for key, value in ctx['config']['models']['yolo']['variants'].items()]))

/home/junior/visual_semantic_navigation/.venv-1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda


,detector,checkpoint
0,yolov8n,experiments/yolov8n.pt
1,yolo26n,experiments/yolo26n.pt


## Detección y recuperación informada por objetos

In [2]:
results = run_detector_benchmark(ctx)
display(results['detector_summary'])
display(results['object_average_precision'])
display(results['object_retrieval_summary'])

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 6611.17it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 6445.07it/s]

,detector,checkpoint,checkpoint_size_mb,model_load_s,ap_min_confidence,operational_confidence,mean_average_precision,tp,fp,fn,precision,recall,f1,mean_detection_ms,std_detection_ms,benchmark_samples
0,yolov8n,/home/junior/visual_semantic_navigation/experi...,6.246372,5.397069,0.001,0.4,0.306349,2759,4018,8035,0.407112,0.255605,0.314040,9.40189,0.496799,20
1,yolo26n,/home/junior/visual_semantic_navigation/experi...,5.287602,6.223347,0.001,0.4,0.324852,2743,3150,8051,0.465468,0.254123,0.328759,12.08492,1.110864,20


,detector,class,average_precision,n_ground_truth,n_predictions
0,yolov8n,bed,0.458670,536,6341
1,yolov8n,book,0.092767,1024,82123
2,yolov8n,chair,0.333403,6257,115080
3,yolov8n,couch,0.152262,386,6664
4,yolov8n,dining table,0.225606,2080,29882
5,yolov8n,keyboard,0.436576,232,4866
6,yolov8n,microwave,0.480154,49,2305
7,yolov8n,sink,0.355203,138,12908
8,yolov8n,toilet,0.467352,64,2901
9,yolov8n,tv,0.061493,28,11332


,dataset_id,method,query_type,language,n_queries,n_negative,recall_at_1,recall_at_3,recall_at_5,mean_reciprocal_rank,mean_rank_first_valid,negative_rejection_rate,room_false_positive_rate,mean_retrieval_latency_ms
0,sunrgbd,siglip_v2_single_view,functional,en,1,0,0.000000,0.00,0.00,0.083333,12.000000,NaN,NaN,32.976861
1,sunrgbd,siglip_v2_single_view,functional,es,1,0,0.000000,0.00,0.00,0.076923,13.000000,NaN,NaN,34.120740
2,sunrgbd,siglip_v2_single_view,multi_object,en,4,0,0.750000,0.75,0.75,0.791667,2.250000,NaN,NaN,33.644188
3,sunrgbd,siglip_v2_single_view,multi_object,es,4,0,0.250000,1.00,1.00,0.583333,2.000000,NaN,NaN,33.812177
4,sunrgbd,siglip_v2_single_view,object,en,3,0,1.000000,1.00,1.00,1.000000,1.000000,NaN,NaN,33.716446
5,sunrgbd,siglip_v2_single_view,object,es,3,0,0.666667,1.00,1.00,0.777778,1.666667,NaN,NaN,33.003005
6,sunrgbd,siglip_v2_yolo26n_objects,functional,en,1,0,0.000000,1.00,1.00,0.333333,3.000000,NaN,NaN,150.501529
7,sunrgbd,siglip_v2_yolo26n_objects,functional,es,1,0,0.000000,1.00,1.00,0.333333,3.000000,NaN,NaN,154.913605
8,sunrgbd,siglip_v2_yolo26n_objects,multi_object,en,4,0,0.000000,0.50,0.50,0.264706,7.000000,NaN,NaN,152.873058
9,sunrgbd,siglip_v2_yolo26n_objects,multi_object,es,4,0,0.000000,0.50,0.75,0.313889,6.750000,NaN,NaN,151.518275


## Relaciones espaciales

La precisión frente a Visual Genome no equivale a una tasa directa de error: el conjunto solo anota relaciones salientes, mientras que las reglas emiten relaciones geométricas exhaustivas. La exhaustividad mide la cobertura de lo anotado.

In [3]:
display(results['relation_metrics'])
display(results['relation_examples'])

,predicate,tp,fp,fn,precision,recall,f1
0,ABOVE,21597,9327952,1220,0.002310,0.946531,0.004609
1,BELOW,20264,9329285,1566,0.002167,0.928264,0.004325
2,LEFT_OF,377,11929005,13,0.000032,0.966667,0.000063
3,NEAR,59247,18994713,245,0.003109,0.995882,0.006200
4,OVERLAPS,11,5603218,3,0.000002,0.785714,0.000004
5,POSSIBLY_ON_TOP_OF,500550,3078007,220391,0.139875,0.694301,0.232841
6,RIGHT_OF,224,16606402,5,0.000013,0.978166,0.000027


,image_id,predicted,annotated,mean_inferred_confidence
0,1,748,11,0.679012
1,2,236,6,0.634357
2,3,2991,23,0.666552
3,4,413,8,0.682759
4,5,2808,22,0.628992
5,6,3551,24,0.646195
6,7,6674,34,0.673669
7,8,4030,28,0.653169
8,9,4313,28,0.669461
9,10,11390,45,0.690813


## Exportación reproducible y figuras

In [4]:
from reproducibility import collect_manifest, save_manifest
results_root = resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'detector_comparison'
figures_root = results_root / 'figures'
results_root.mkdir(parents=True, exist_ok=True)
for name, frame in results.items():
    frame.to_csv(results_root / f'{name}.csv', index=False)
figure_paths = create_detector_figures(results, figures_root)
vlm_root = resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'vlm_comparison'
vlm_results = {name: pd.read_csv(vlm_root / f'{name}.csv')
               for name in ('cases', 'summary', 'paired_differences',
                            'threshold_diagnostics', 'model_costs')}
summary_path = write_results_summary(
    resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'OFFLINE_RESULTS.md',
    vlm_results, results)
manifest = collect_manifest(ctx['config'], repo_dir=str(ctx['repo_root']), device=ctx['device'],
    extra={'notebook': '02_yolo_and_relations',
           'detectors': ctx['config']['models']['yolo']['variants'],
           'object_retrieval_encoder': ctx['config']['models']['siglip']['object_retrieval_variant'],
           'figures': figure_paths})
save_manifest(str(results_root / 'manifest.json'), manifest)
print(f'Resultados: {results_root}')
print(f'Resumen consolidado: {summary_path}')
print('Figuras generadas:')
for path in figure_paths:
    print(' -', path)

Resultados: /home/junior/visual_semantic_navigation/experiments/offline/results/detector_comparison
Resumen consolidado: /home/junior/visual_semantic_navigation/experiments/offline/results/OFFLINE_RESULTS.md
Figuras generadas:
 - /home/junior/visual_semantic_navigation/experiments/offline/results/detector_comparison/figures/detectores_ap_por_clase.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/detector_comparison/figures/detectores_ap_por_clase.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/detector_comparison/figures/detectores_metricas_globales.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/detector_comparison/figures/detectores_metricas_globales.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/detector_comparison/figures/objetos_efecto_en_recuperacion.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/detector_comparison/figures/objetos_efecto_en_recu

## Límite de interpretación

Este cuaderno no evalúa multivista, contaminación entre habitaciones, política espacial ni éxito de navegación. Tampoco puede estimar offline la aportación de las relaciones a la recuperación porque el banco de consultas disponible no contiene ground truth relacional independiente. Esos factores quedan reservados a simulación.